# 05. Sessions

**Session**은 여러 번의 `Runner.run` 호출에 걸쳐 **대화 히스토리를 자동으로 유지**하는 기능입니다.
- 세션을 쓰지 않으면 매 턴마다 이전 메시지 목록을 직접 구성해 넘겨야 합니다.
- `session`을 넘기면 SDK가 대화를 저장소에 저장/불러와서, 같은 `session`으로 실행할 때마다 맥락이 이어집니다.

활용: 채팅봇, 고객지원 에이전트 등 다회차 대화.

### SDK 제공 Session 유형 (10종)

| 세션 | 위치 | 특징 | 용도 | 본 노트북 실습 |
|---|---|---|---|---|
| `SQLiteSession` | `agents` | 기본 내장, 인메모리/파일 저장 | 개발/프로토타입 | ✅ |
| `AsyncSQLiteSession` | `agents.extensions.memory` | 비동기 전용 SQLite | 비동기 환경 | ✅ |
| `AdvancedSQLiteSession` | `agents.extensions.memory` | 브랜치, 토큰 사용량 분석 | 고급 분석 필요 시 | ✅ |
| `SQLAlchemySession` | `agents.extensions.memory` | PostgreSQL, MySQL 등 지원 | 프로덕션 DB | ✅ |
| `EncryptedSession` | `agents.extensions.memory` | 다른 세션을 암호화로 래핑 | 보안 필요 시 | ✅ |
| `OpenAIConversationsSession` | `agents` | OpenAI 서버에 대화 저장 | OpenAI 플랫폼 활용 | ✅ |
| `CustomSession` | 직접 구현 | `Session` 프로토콜 구현 | Django, MongoDB 등 연동 | ✅ |
| `RedisSession` | `agents.extensions.memory` | Redis 기반 분산 저장 | 대규모 서비스 | ❌ (Redis 서버 필요) |
| `MongoDBSession` | `agents.extensions.memory` | MongoDB 도큐먼트 저장 | NoSQL 스택 | ❌ (MongoDB 필요) |
| `DaprSession` | `agents.extensions.memory` | Dapr state store 추상화 | K8s/클라우드 | ❌ (Dapr 사이드카 필요) |

### 학습 내용

1. **세션 없이 실행** — 이전 대화 맥락이 유지되지 않는 문제 확인
2. **`SQLiteSession`** — 대화 이어가기, 세션 ID별 대화 분리
3. **추가 인프라 없이 실행 가능한 6종 실습** — Async/Advanced SQLite, SQLAlchemy(프로덕션 DB),
   Encrypted(암호화), OpenAIConversations(서버 저장), Custom(직접 구현)
4. **종합 예제: Session으로 챗봇 만들기** — `while True` + `input()` 챗봇 구현,
   세션 교체만으로 저장 방식이 바뀌는 것 확인
5. **실습 문제** — 본문 챗봇을 다른 Session으로 응용 (보안 챗봇, 상담 리포트 챗봇)

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import os
from agents import Agent, Runner, SQLiteSession

Model = os.environ["OPENAI_MODEL"]

## 세션 없이 실행 (맥락 없음)

세션 없이 두 번 호출하면, 두 번째 질문에 대한 맥락(이전 대화 내역과 연관된)이 없습니다.

In [3]:
agent = Agent(
    name="Assistant",
    instructions="간결하게 답변해 주세요.",
    model=Model
)

In [4]:
question_1 = "남산 팔각정은 어느 도시에 있나요?"
result1 = await Runner.run(agent, question_1)
print("Q1:", question_1)
print("A1:", result1.final_output)

Q1: 남산 팔각정은 어느 도시에 있나요?
A1: 남산 팔각정은 **서울특별시**에 있습니다.


In [5]:
question_2 = "그 도시는 어느 나라에 있나요?"
result2 = await Runner.run(agent, question_2)
print("Q2:", question_2)
print("A2:", result2.final_output)

Q2: 그 도시는 어느 나라에 있나요?
A2: 어느 도시인지 말씀해 주시면 나라를 알려드릴게요.


## SQLiteSession으로 대화 이어가기

같은 `session` 인스턴스를 넘기면, 이전 턴의 메시지가 자동으로 포함됩니다.

In [6]:
session = SQLiteSession("conversation_123")

In [7]:
question_1 = "남산 팔각정은 어느 도시에 있나요?"
result1 = await Runner.run(
    agent, 
    question_1,
    session=session,
)
print("Q1:", question_1)
print("A1:", result1.final_output)

Q1: 남산 팔각정은 어느 도시에 있나요?
A1: 남산 팔각정은 **서울특별시**에 있습니다.


In [8]:
question_2 = "그 도시는 어느 나라에 있나요?"
result2 = await Runner.run(
    agent, 
    question_2,
    session=session
)
print("Q2:", question_2)
print("A2:", result2.final_output)

Q2: 그 도시는 어느 나라에 있나요?
A2: 서울특별시는 **대한민국**에 있습니다.


In [9]:
question_3 = "그곳 인구는 얼마인가요?"
result3 = await Runner.run(
    agent, 
    question_3,
    session=session
)
print("Q3:", question_3)
print("A3:", result3.final_output)

Q3: 그곳 인구는 얼마인가요?
A3: 대한민국의 인구는 약 **5,100만 명**입니다.


## 세션 ID별로 대화 분리

`session_id`가 다르면 서로 다른 대화로 저장됩니다.
예: `SQLiteSession("user_456", "conversations.db")`로 사용자/채널별 분리.

In [10]:
session_b = SQLiteSession("another_user_345")
result_b = await Runner.run(agent, "그곳 인구는 얼마인가요?", session=session_b)
print("User B - A1:", result_b.final_output)

User B - A1: “그곳”이 어디를 뜻하는지 알려주시면 인구를 찾아드릴게요.


---------------------
## SDK가 제공하는 Session 유형

아래 6가지는 **추가 인프라 없이** 바로 실행할 수 있습니다.
(`RedisSession`, `MongoDBSession`, `DaprSession`은 각각 Redis/MongoDB/Dapr 서버가 필요해 제외)

| 실습 순서 | Session | 시연 포인트 |
|---|---|---|
| 1 | `AsyncSQLiteSession` | 비동기 전용 SQLite |
| 2 | `AdvancedSQLiteSession` | 턴별 토큰 사용량 분석 |
| 3 | `SQLAlchemySession` | URL만 바꾸면 PostgreSQL/MySQL로 교체 가능 |
| 4 | `EncryptedSession` | 저장 내용 암호화 (원본 DB에는 암호문만 저장) |
| 5 | `OpenAIConversationsSession` | 대화를 OpenAI 서버에 저장 |
| 6 | `CustomSession` | `Session` 프로토콜 직접 구현 |

---------------------
## 종합 예제: Session으로 챗봇 만들기

Session의 대표 활용처는 **챗봇**입니다. 구조는 아주 단순합니다.

```
while True:
    input()으로 사용자 입력 받기       ← "종료"면 break
    Runner.run(agent, 입력, session)   ← 같은 session 반복 사용 → 맥락 자동 유지
    답변 출력
```

- 핵심은 **같은 `session`을 루프 안에서 반복 사용**하는 것입니다. 모델 자체는 기억이 없고,
  세션이 이전 대화를 붙여주기 때문에 챗봇이 "기억하는 것처럼" 동작합니다.
- **사용자 ID = 세션 ID**로 하면 사용자별로 대화가 분리됩니다.
- 아래 `run_chatbot()`은 `session` 인자로 **어떤 Session이든 갈아끼울 수 있게** 만들었습니다.
  (인자를 생략하면 기본값 `SQLiteSession(user_id, "chatbot.db")` 사용)

> `input()`은 노트북 자동 실행 시 멈추기 때문에, `inputs` 리스트를 주면 **자동 데모**,
> 주지 않으면 `input()`으로 **직접 대화**하는 겸용 구조로 작성했습니다.

In [11]:
chatbot = Agent(
    name="Chatbot",
    instructions="친절한 상담 챗봇입니다. 간결하게 답변하세요.",
    model=Model,
)

async def run_chatbot(user_id: str, session=None, inputs: list | None = None):
    """while True 챗봇 루프.

    - session : 사용할 Session (생략 시 SQLiteSession(user_id, "chatbot.db"))
    - inputs=None  : input()으로 직접 대화 (터미널/Jupyter에서 직접 실행)
    - inputs=[...] : 리스트에서 순서대로 입력을 가져오는 자동 데모 (리스트 소진 시 자동 종료)
    """
    if session is None:
        session = SQLiteSession(user_id, "chatbot.db")   # 기본: 사용자 ID = 세션 ID
    scripted = iter(inputs) if inputs is not None else None

    print(f"=== {user_id}님과의 대화 시작 ('종료' 입력 시 끝) ===")
    while True:
        if scripted is None:
            message = input(f"{user_id}: ")              # 직접 입력
        else:
            message = next(scripted, "종료")             # 자동 데모 입력
            print(f"{user_id}: {message}")

        if message.strip() == "종료":
            print("챗봇을 종료합니다.\n")
            break

        result = await Runner.run(chatbot, message, session=session)
        print("챗봇:", result.final_output)

### 예제 1: 다중 사용자 챗봇 (SQLiteSession)

- 사용자마다 다른 세션 → **대화가 섞이지 않음**
- 파일 DB(`chatbot.db`) → `"종료"` 후 **재접속해도 대화가 이어짐**

In [12]:
# 재실행 시 이전 기록이 누적되지 않도록 초기화 
await SQLiteSession("철수", "chatbot.db").clear_session()
await SQLiteSession("영희", "chatbot.db").clear_session()

# 철수 접속 → 자기소개 → 종료
await run_chatbot("철수", inputs=["저는 부산에 살아요."])

# 영희 접속 → 자기소개 → 종료
await run_chatbot("영희", inputs=["저는 제주에 살아요."])

# 철수 재접속 → 파일 DB 덕분에 이전 대화가 이어짐 (부산이라고 답해야 정상)
await run_chatbot("철수", inputs=["제가 어디 산다고 했죠?"])

# 영희 재접속 → 제주라고 답해야 정상 (철수의 대화와 섞이지 않음)
await run_chatbot("영희", inputs=["제가 어디 산다고 했죠?"])

=== 철수님과의 대화 시작 ('종료' 입력 시 끝) ===
철수: 저는 부산에 살아요.
챗봇: 반갑습니다. 부산에 사시는군요.  
무엇을 도와드릴까요?
철수: 종료
챗봇을 종료합니다.

=== 영희님과의 대화 시작 ('종료' 입력 시 끝) ===
영희: 저는 제주에 살아요.
챗봇: 좋네요. 제주에 사신다니 부럽습니다.  
제주에서 생활 관련, 여행, 맛집, 날씨, 지역 정보까지 도와드릴게요.
영희: 종료
챗봇을 종료합니다.

=== 철수님과의 대화 시작 ('종료' 입력 시 끝) ===
철수: 제가 어디 산다고 했죠?
챗봇: 부산에 산다고 하셨어요.
철수: 종료
챗봇을 종료합니다.

=== 영희님과의 대화 시작 ('종료' 입력 시 끝) ===
영희: 제가 어디 산다고 했죠?
챗봇: 제주에 산다고 하셨어요.
영희: 종료
챗봇을 종료합니다.



In [13]:
# 자유 대화
await run_chatbot("Anyone")

=== Anyone님과의 대화 시작 ('종료' 입력 시 끝) ===


Anyone:  안녕, 난 길동이야


챗봇: 안녕하세요, 길동님! 반갑습니다.


Anyone:  내 이름이 뭐였지?


챗봇: 길동님입니다.


Anyone:  종료


챗봇을 종료합니다.

